# Germanene bubble: height sweep at radius 3 nm

Compare peak heights **0.5, 1.0, 1.5 and 2.0 nm**, changing only the Gaussian height in the existing TBPLaS model:

$$z(X,Y)=h\exp\left[-\frac{X^2+Y^2}{2(R/3)^2}\right],\qquad R=3\,\mathrm{nm}.$$

The radius convention is **three Gaussian standard deviations**, so the width is 1 nm. The sheet is prescribed by this profile; no mechanical relaxation is performed. Material parameters, global pz axes and fixed SOC are inherited from `model.py`.

The default `lfac=3` follows the current geometry and line-LDOS notebooks (01 and 02), giving an approximately 18 × 18 nm periodic cell. Set `LFAC=5` to use the larger cell of notebooks 03 and 04. Comparisons within this notebook always use identical cells, site selections, broadening and recursion settings. This is a parameter sweep of the existing model, not a finite-size or model-validity study.

Full settings: 25 A+B line positions, both spins averaged, 1201 energies from −1 to +1 eV, 10 meV broadening, 1500 recursion steps and eight random vectors. The A+B bubble ROI and full-cell DOS use matched random phases for all heights and the flat reference. All densities are normalized per spin orbital, following the existing notebooks.

Run all cells. `HB_QUICK=1` uses reduced recursion and sampling for a smoke check; outputs go to a separate `quick` folder. Full calculations may take several minutes.

Intrinsic Rashba is explicitly disabled in this sweep: $\lambda_{R2}=0$, while $\lambda_{\mathrm{SO}}=100\,\mathrm{meV}$ remains unchanged. All saved filenames begin with `H_`. Figures use 300 DPI and lettered panels.

In [ ]:
# Resolve the shared model and output root from this notebook or the project root.
from pathlib import Path
import os
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'model.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import os
import json
import csv
from dataclasses import asdict, replace
from pathlib import Path
from time import perf_counter
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from model import (ExportParams, build_model, cell_display, pseudofield,
                   gaussian_height, line_sites, line_ldos, bubble_sites,
                   roi_ldos, total_dos)

fig_size0 = (6.8, 4.2)

params = {
    "backend": "ps",
    "axes.labelsize": 13,
    "font.size": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "text.usetex": True,
    "figure.figsize": fig_size0,
    "font.family": "serif",
    "font.serif": "Computer Modern Roman",
    "legend.frameon": True,
    "savefig.dpi": 300,
}

plt.rcParams.update(params)


def label_panels(axes):
    """Label data panels left-to-right, then top-to-bottom; omit colorbars."""
    panels = np.asarray(axes, dtype=object).ravel()
    if len(panels) > 1:
        for index, ax in enumerate(panels):
            ax.set_title(f'({chr(ord("a") + index)})', loc='left',
                         fontsize=13, fontweight='bold')

HEIGHTS_NM = np.array([0.5, 1.0, 1.5, 2.0])
RADIUS_NM = 3.0
LFAC = 3.0
QUICK = os.getenv('HB_QUICK', '0') == '1'
SEED = 20260923
base = ExportParams(radius=RADIUS_NM, lfac=LFAC, lambda_r2_mev=0.0)
energy = base.energy
depth = 250 if QUICK else base.recursion_depth
points = 5 if QUICK else 25
draws = 2 if QUICK else base.roi_random_vectors
out = Path('results/gaussian/05_height_sweep') / ('quick' if QUICK else 'full')
out.mkdir(parents=True, exist_ok=True)
metadata = dict(base_parameters=asdict(base), heights_nm=HEIGHTS_NM.tolist(),
                recursion_depth=depth, line_points=points, random_vectors=draws,
                seed=SEED, quick=QUICK, radius_convention='R = 3 sigma',
                roi_sublattices='both', normalization='density per spin orbital')
(out / 'H_parameters.json').write_text(json.dumps(metadata, indent=2) + '\n')
print('QUICK CHECK' if QUICK else 'FULL SPECTRAL SETTINGS')
print(f'R = {RADIUS_NM:g} nm; heights = {HEIGHTS_NM} nm; lfac = {LFAC:g}')
print(f'{depth} steps; {points} line positions; {draws} random vectors')

## Common flat reference and sampling
The same physical atom indices and random phases are used throughout. The flat reference is computed once. The ROI difference uncertainty uses paired random-vector differences.

In [ ]:
t0 = perf_counter()
flat = build_model(base, flat=True)
ham_flat = flat.hamiltonian()
position, sites, targets = line_sites(flat, length=9.0, points=points,
                                     angle_deg=0, sublattice='both')
roi = bubble_sites(flat)
assert np.any(flat.is_a[roi]) and np.any(~flat.is_a[roi])
print(f'ROI: {np.count_nonzero(flat.is_a[roi])} A + {np.count_nonzero(~flat.is_a[roi])} B atoms')
phases = np.exp(2j * np.pi * np.random.default_rng(SEED).random((len(roi), draws)))
flat_line = line_ldos(ham_flat, sites, energy, base.eta, depth)
flat_roi, flat_roi_error, flat_roi_draws = roi_ldos(
    ham_flat, roi, energy, base.eta, depth, phases)
flat_dos, flat_dos_error, flat_dos_draws = total_dos(
    ham_flat, energy, base.eta, depth, draws=draws, seed=SEED)
np.savez_compressed(out / 'H_flat_reference.npz', energy=energy, position=position,
                    sites=sites, targets=targets, roi=roi, roi_is_a=flat.is_a[roi], phases=phases,
                    display_xy=flat.display_xy, line_ldos=flat_line,
                    roi_ldos=flat_roi, roi_error=flat_roi_error,
                    roi_draws=flat_roi_draws, dos=flat_dos,
                    dos_error=flat_dos_error, dos_draws=flat_dos_draws)
print(f'{flat.dim[0]} × {flat.dim[1]} cells; {ham_flat.shape[0]:,} spin orbitals; '
      f'{len(roi)} ROI atoms; flat reference: {perf_counter()-t0:.1f} s')
del ham_flat

## Calculate the four heights
Each height is saved separately as soon as it finishes. Field maps retain both PMF conventions from notebook 01. The main comparison displays Eq. (A11) as printed, as in notebook 02.

In [ ]:
results = []
for height_nm in HEIGHTS_NM:
    start = perf_counter()
    p = replace(base, height=float(height_nm))
    bubble = build_model(p)
    ham = bubble.hamiltonian()
    residual = ham - ham.getH()
    assert residual.nnz == 0 or np.max(np.abs(residual.data)) < 1e-12
    pos, selected, _ = line_sites(bubble, length=9.0, points=points,
                                  angle_deg=0, sublattice='both')
    np.testing.assert_array_equal(selected, sites)
    np.testing.assert_allclose(pos, position)
    np.testing.assert_array_equal(bubble_sites(bubble), roi)
    gauge = pseudofield(bubble)
    rho_line = line_ldos(ham, sites, energy, p.eta, depth)
    rho_roi, roi_error, roi_draws = roi_ldos(ham, roi, energy, p.eta, depth, phases)
    dos, dos_error, dos_draws = total_dos(ham, energy, p.eta, depth,
                                        draws=draws, seed=SEED)
    paired = roi_draws - flat_roi_draws
    result = dict(roi_count=len(roi), roi_a_count=int(np.count_nonzero(bubble.is_a[roi])),
                  roi_b_count=int(np.count_nonzero(~bubble.is_a[roi])),
                  roi_is_a=bubble.is_a[roi], height_nm=float(height_nm), x=gauge['x'], y=gauge['y'],
                  height=cell_display(bubble, bubble.height),
                  strain=cell_display(bubble, bubble.strain),
                  Bpaper=gauge['Bpaper'], Bconsistent=gauge['B'],
                  line_ldos=rho_line, roi_ldos=rho_roi, roi_error=roi_error,
                  roi_draws=roi_draws, roi_difference=paired.mean(axis=1),
                  roi_difference_error=paired.std(axis=1, ddof=1)/np.sqrt(draws),
                  dos=dos, dos_error=dos_error, dos_draws=dos_draws)
    for key, value in result.items():
        assert np.all(np.isfinite(value)), f'Non-finite {key} at h={height_nm}'
    for key in ('line_ldos', 'roi_ldos', 'dos'):
        assert np.min(result[key]) >= -1e-12, f'Negative density: {key}'
    tag = f'h{height_nm:.1f}nm'.replace('.', 'p')
    np.savez_compressed(out / f'H_{tag}.npz', **result, energy=energy,
                        position=position, sites=sites, roi=roi,
                        parameters_json=json.dumps(asdict(p)), roi_sublattices='both', quick=QUICK,
                        recursion_depth=depth, random_vectors=draws)
    results.append(result)
    print(f'h = {height_nm:g} nm finished in {perf_counter()-start:.1f} s', flush=True)
    del ham, bubble
print(f'All calculations finished in {(perf_counter()-t0)/60:.1f} min')

## Geometry and PMF on common colour scales
Maps are averages over the four physical atoms in each rectangular cell. Each row uses one scale across all heights, without percentile clipping. The circle marks radius 3 nm.

In [ ]:
def map_extent(result):
    x, y = result['x'], result['y']
    dx, dy = np.diff(x).mean(), np.diff(y).mean()
    return [x[0]-dx/2, x[-1]+dx/2, y[0]-dy/2, y[-1]+dy/2]

fig, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
for row, (key, label, cmap) in enumerate([
        ('height', 'height (nm)', 'viridis'),
        ('strain', 'bond strain', 'magma'),
        ('Bpaper', 'PMF (T)', 'RdBu_r')]):
    vmin = min(np.min(r[key]) for r in results)
    vmax = max(np.max(r[key]) for r in results)
    if key == 'Bpaper':
        vmax = max(abs(vmin), abs(vmax))
        vmin = -vmax
    for col, r in enumerate(results):
        ax = axes[row, col]
        im = ax.imshow(r[key], origin='lower', extent=map_extent(r),
                       cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
        ax.add_patch(plt.Circle((0, 0), RADIUS_NM, fill=False, color='darkorange'))
        ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), aspect='equal',
               xlabel='x (nm)', ylabel='y (nm)', title=f"h = {r['height_nm']:g} nm")
    fig.colorbar(im, ax=axes[row, :].tolist(), label=label, shrink=0.8)
label_panels(axes)
fig.savefig(out / 'H_geometry_pmf.png', dpi=300)
plt.show()

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(results)))
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
x_profile = np.linspace(-4.5, 4.5, 501)
for r, color in zip(results, colors):
    ax.plot(x_profile, gaussian_height(x_profile, 0, replace(base, height=r['height_nm'])),
            color=color, label=f"h = {r['height_nm']:g} nm")
ax.set(xlabel='x (nm)', ylabel='height (nm)', title='Gaussian profiles along y = 0')
ax.legend()
fig.savefig(out / 'H_height_profiles.png', dpi=300)
plt.show()


## Line spectroscopy
Each spectrum averages the nearest A and B sites and their two spin projections. All maps share the same log colour scale. Positions run from −4.5 to +4.5 nm at Y = 0.

The spatial trace is evaluated at $E=+0.1\,\mathrm{eV}$ (the positive flat-sheet band edge for the chosen intrinsic SOC), using the nearest saved energy-grid point for both bubble and flat curves.

In [ ]:
logs = [np.log10(np.maximum(r['line_ldos'], 1e-12)) for r in results]
vmin, vmax = min(z.min() for z in logs), max(z.max() for z in logs)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True, constrained_layout=True)
for ax, r, z in zip(axes, results, logs):
    im = ax.pcolormesh(position, energy, z, shading='nearest',
                       cmap='magma', vmin=vmin, vmax=vmax, rasterized=True)
    ax.set(xlabel='X (nm)', title=f"h = {r['height_nm']:g} nm")
axes[0].set_ylabel('energy (eV)')
fig.colorbar(im, ax=axes.tolist(), label='log10(LDOS / (1/eV))')
label_panels(axes)
fig.savefig(out / 'H_line_ldos_maps.png', dpi=300)
plt.show()

centre = int(np.argmin(abs(position)))
zero = int(np.argmin(abs(energy)))  # retained for the E=0 CSV summary
TRACE_ENERGY_EV = 0.1
trace_index = int(np.argmin(abs(energy - TRACE_ENERGY_EV)))
trace_energy = float(energy[trace_index])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for r, color in zip(results, colors):
    label = f"h = {r['height_nm']:g} nm"
    axes[0].plot(energy, r['line_ldos'][:, centre], color=color, label=label)
    axes[1].plot(energy, r['line_ldos'][:, 0], color=color, label=label)
    axes[2].plot(position, r['line_ldos'][trace_index], '.-', color=color, label=label)
for ax, index, title in zip(axes[:2], [centre, 0], ['bubble centre', 'far endpoint']):
    ax.plot(energy, flat_line[:, index], 'k--', lw=1, label='flat reference')
    ax.set(xlabel='energy (eV)', ylabel='LDOS (1/eV)', title=title)
axes[2].plot(position, flat_line[trace_index], 'k--', lw=1, label='flat reference')
axes[2].set(xlabel='X (nm)', ylabel=f'LDOS at E = {trace_energy:g} eV (1/eV)', title='band-edge trace')
axes[0].legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'H_local_spectra.png', dpi=300)
plt.show()

## Bubble-region spectra and full-cell DOS
The ROI contains all A and B atoms within radius 3 nm at every height (198 A + 198 B = 396 atoms for the default geometry). Both spin projections are averaged at each atom; the spectrum is an average per spin orbital, so including B does not double its normalization. Shading shows one standard error from the random-vector samples. The bubble-minus-flat error is estimated from **paired differences**; it does not measure recursion or finite-size error.

As in notebook 06, the first figure shows a separate bubble-versus-flat panel for each height, with a shared vertical scale. The second figure overlays the heights for the bubble-minus-flat spectrum and full-cell DOS. Figures are saved as `H_roi_spectra.png` and `H_roi_difference_dos.png`.

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 4.5),
                         sharey=True, squeeze=False, constrained_layout=True)
for ax, r, color in zip(axes.ravel(), results, colors):
    ax.plot(energy, r['roi_ldos'], color=color, label='bubble')
    ax.fill_between(energy, r['roi_ldos']-r['roi_error'],
                    r['roi_ldos']+r['roi_error'], color=color, alpha=0.15)
    ax.plot(energy, flat_roi, 'k--', lw=1, label='flat')
    ax.fill_between(energy, flat_roi-flat_roi_error,
                    flat_roi+flat_roi_error, color='0.5', alpha=0.1)
    ax.set(xlabel='energy (eV)',
           title=f"h = {r['height_nm']:g} nm")
axes[0,0].set_ylabel('mean DOS (1/eV)')
axes[0,0].legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'H_roi_spectra.png', dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for r, color in zip(results, colors):
    for ax, key, error in zip(axes, ['roi_difference', 'dos'],
                              ['roi_difference_error', 'dos_error']):
        ax.plot(energy, r[key], color=color, label=f"h = {r['height_nm']:g} nm")
        ax.fill_between(energy, r[key]-r[error], r[key]+r[error], color=color, alpha=0.12)
axes[0].axhline(0, color='0.5', lw=0.7)
axes[1].plot(energy, flat_dos, 'k--', lw=1, label='flat reference')
axes[0].set(xlabel='energy (eV)', ylabel='mean LDOS difference (1/eV)',
            title='bubble minus flat')
axes[1].set(xlabel='energy (eV)', ylabel='DOS per spin orbital (1/eV)',
            title='full-cell DOS: fixed cell')
for ax in axes:
    ax.legend(fontsize=8)
label_panels(axes)
fig.savefig(out / 'H_roi_difference_dos.png', dpi=300)
plt.show()

## Saved numerical summary
PMF extrema below refer to cell-averaged maps. Arrays, stochastic samples, physical site indices and parameters are saved alongside the figures so the comparisons can be reproduced.

In [ ]:
summary = []
for r in results:
    summary.append(dict(height_nm=r['height_nm'], radius_nm=RADIUS_NM,
                        roi_atoms=r['roi_count'], roi_a_atoms=r['roi_a_count'],
                        roi_b_atoms=r['roi_b_count'],
                        max_cell_strain=float(np.max(r['strain'])),
                        max_abs_Bpaper_T=float(np.max(np.abs(r['Bpaper']))),
                        max_abs_Bconsistent_T=float(np.max(np.abs(r['Bconsistent']))),
                        centre_ldos_E0=float(r['line_ldos'][zero, centre]),
                        far_ldos_E0=float(r['line_ldos'][zero, 0]),
                        roi_ldos_E0=float(r['roi_ldos'][zero])))
with (out / 'H_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
print('height (nm) | max cell strain | max |Bpaper| (T) | centre LDOS(E=0) (1/eV)')
for row in summary:
    print(f"{row['height_nm']:11.1f} | {row['max_cell_strain']:15.5f} | "
          f"{row['max_abs_Bpaper_T']:16.3f} | {row['centre_ldos_E0']:.8f}")
print(f'Figures, arrays, parameters and CSV saved in {out.resolve()}')